# Silver layer: Bronze → Silver (PySpark)
Reads raw CSV files from the `bronze` container, cleans/transforms them, and writes Parquet to the `silver` container.

**Before running:** set `STORAGE_ACCOUNT` and `SECRET_SCOPE` below and store the service-principal credentials as Databricks secrets. Never hard-code them in the notebook.

In [ ]:
from pyspark.sql.functions import (
    col, lit, concat, concat_ws, when, split,
    to_date, datediff, dayofmonth, month, year,
)

## Data access (service principal / OAuth)

In [ ]:
STORAGE_ACCOUNT = "<storage-account>"
SECRET_SCOPE = "<secret-scope>"  # Databricks secret scope (ideally backed by Azure Key Vault)

client_id = dbutils.secrets.get(scope=SECRET_SCOPE, key="sp-client-id")
client_secret = dbutils.secrets.get(scope=SECRET_SCOPE, key="sp-client-secret")
tenant_id = dbutils.secrets.get(scope=SECRET_SCOPE, key="tenant-id")

suffix = f"{STORAGE_ACCOUNT}.dfs.core.windows.net"
spark.conf.set(f"fs.azure.account.auth.type.{suffix}", "OAuth")
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{suffix}",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
)
spark.conf.set(f"fs.azure.account.oauth2.client.id.{suffix}", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{suffix}", client_secret)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{suffix}",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
)

BRONZE = f"abfss://bronze@{suffix}"
SILVER = f"abfss://silver@{suffix}"

## Helpers

In [ ]:
def read_bronze(folder):
    """Read a raw CSV dataset from the bronze container."""
    return (
        spark.read.format("csv")
        .option("header", "true")
        .load(f"{BRONZE}/{folder}/")
    )


def write_silver(df, folder):
    """Write a dataset to the silver container as Parquet (overwrite = safe to re-run)."""
    df.write.format("parquet").mode("overwrite").save(f"{SILVER}/{folder}")

## Transformations

### Calendar

In [ ]:
df_calendar = read_bronze("AdventureWorks_Calendar")

df_calendar = (
    df_calendar
    .withColumn("Date", to_date("Date", "M/d/yyyy"))
    .withColumn("Day", dayofmonth("Date"))
    .withColumn("Month", month("Date"))
    .withColumn("Year", year("Date"))
)

write_silver(df_calendar, "AdventureWorks_Calendar")

### Customers

In [ ]:
df_customers = read_bronze("AdventureWorks_Customers")

df_customers = (
    df_customers
    # concat_ws skips NULLs (concat would return NULL if any part is missing)
    .withColumn("Full_Name", concat_ws(" ", "Prefix", "FirstName", "LastName"))
    # Income bands: High >= 70,000 | Low <= 30,000 | Medium in between
    .withColumn(
        "Income Type",
        when(col("AnnualIncome").cast("int") >= 70000, lit("High"))
        .when(col("AnnualIncome").cast("int") <= 30000, lit("Low"))
        .otherwise(lit("Medium")),
    )
)

write_silver(df_customers, "AdventureWorks_Customers")

### Product categories

In [ ]:
df_product_cat = read_bronze("AdventureWorks_Product_Categories")
write_silver(df_product_cat, "AdventureWorks_Product_Categories")

### Product subcategories

In [ ]:
df_product_subcat = read_bronze("AdventureWorks_Product_Subcategories")
write_silver(df_product_subcat, "AdventureWorks_Product_Subcategories")

### Products

In [ ]:
df_products = read_bronze("AdventureWorks_Products")

df_products = (
    df_products
    .withColumn("Serial No", concat(col("ProductKey"), lit("-"), col("ProductSubcategoryKey")))
    .withColumn("Catagory", split(col("ProductName"), "-")[0])
    .withColumn(
        "ProductSize",
        when(col("ProductSize") == "0", None)  # "0" means no size
        .when(col("ProductSize").cast("int").between(38, 42), "S")
        .when(col("ProductSize").cast("int").between(44, 48), "M")
        .when(col("ProductSize").cast("int").between(52, 56), "L")
        .when(col("ProductSize").cast("int").between(58, 62), "XL")
        .otherwise(col("ProductSize")),
    )
)

write_silver(df_products, "AdventureWorks_Products")

### Returns

In [ ]:
df_returns = read_bronze("AdventureWorks_Returns")

df_returns = df_returns.withColumn("ReturnDate", to_date("ReturnDate", "M/d/yyyy"))

write_silver(df_returns, "AdventureWorks_Returns")

### Sales 2015–2017
The same transformation is applied to all three years:
- `OrderDate` and `StockDate` converted to dates
- `Selling Time(In Days)` = `StockDate` − `OrderDate`

In [ ]:
def transform_sales(df):
    return (
        df
        .withColumn("OrderDate", to_date("OrderDate", "M/d/yyyy"))
        .withColumn("StockDate", to_date("StockDate", "M/d/yyyy"))
        .withColumn("Selling Time(In Days)", datediff(col("StockDate"), col("OrderDate")))
    )


for sales_year in (2015, 2016, 2017):
    folder = f"AdventureWorks_Sales_{sales_year}"
    write_silver(transform_sales(read_bronze(folder)), folder)

### Territories

In [ ]:
df_territories = read_bronze("AdventureWorks_Territories")
write_silver(df_territories, "AdventureWorks_Territories")

## Validation
Row counts of every Silver dataset (quick sanity check that all writes succeeded).

In [ ]:
silver_folders = [
    "AdventureWorks_Calendar", "AdventureWorks_Customers",
    "AdventureWorks_Product_Categories", "AdventureWorks_Product_Subcategories",
    "AdventureWorks_Products", "AdventureWorks_Returns",
    "AdventureWorks_Sales_2015", "AdventureWorks_Sales_2016", "AdventureWorks_Sales_2017",
    "AdventureWorks_Territories",
]

for folder in silver_folders:
    print(f"{folder:45s} {spark.read.parquet(f'{SILVER}/{folder}').count():>8,} rows")